In [2]:
from dataclasses import dataclass
from datetime import date
import math

# -----------------------------
# Produit : EDF Green Bond
# -----------------------------
@dataclass
class GreenBond:
    issuer: str
    isin: str
    face: float
    coupon_rate: float
    maturity_years: float
    frequency: int


bond = GreenBond(
    issuer="EDF",
    isin="FR001400ZGF2",
    face=1000,
    coupon_rate=0.0325,     # 3.25%
    maturity_years=6,       # approx remaining maturity
    frequency=1
)

# -----------------------------
# Cashflows
# -----------------------------
def cashflows(bond):
    cpn = bond.face * bond.coupon_rate / bond.frequency
    flows = []

    for t in range(1, int(bond.maturity_years * bond.frequency) + 1):
        cf = cpn
        if t == bond.maturity_years * bond.frequency:
            cf += bond.face
        flows.append((t/bond.frequency, cf))

    return flows


# -----------------------------
# 1️⃣ Pricing par YTM
# -----------------------------
def price_from_ytm(bond, ytm):

    pv = 0

    for t, cf in cashflows(bond):
        pv += cf / (1 + ytm) ** t

    return pv


# -----------------------------
# 2️⃣ Pricing par courbe zéro-coupon
# -----------------------------
def price_from_spot_curve(bond, spot_curve):

    pv = 0

    for t, cf in cashflows(bond):

        r = spot_curve.get(t, list(spot_curve.values())[-1])

        pv += cf * math.exp(-r * t)

    return pv


# -----------------------------
# 3️⃣ Pricing avec spread de crédit
# -----------------------------
def price_with_credit_spread(bond, risk_free_rate, credit_spread):

    r = risk_free_rate + credit_spread

    pv = 0

    for t, cf in cashflows(bond):
        pv += cf / (1 + r) ** t

    return pv


# -----------------------------
# 4️⃣ Pricing avec greenium
# -----------------------------
def price_with_greenium(bond, risk_free_rate, credit_spread, greenium_bp):

    greenium = greenium_bp / 10000

    adjusted_spread = credit_spread - greenium

    r = risk_free_rate + adjusted_spread

    pv = 0

    for t, cf in cashflows(bond):
        pv += cf / (1 + r) ** t

    return pv


# -----------------------------
# Hypothèses marché
# -----------------------------
ytm = 0.033

spot_curve = {
    1: 0.027,
    2: 0.028,
    3: 0.029,
    4: 0.030,
    5: 0.031,
    6: 0.032
}

risk_free_rate = 0.029
credit_spread = 0.009      # 90 bp
greenium_bp = 15           # 15 bp

# -----------------------------
# Calculs
# -----------------------------
p1 = price_from_ytm(bond, ytm)

p2 = price_from_spot_curve(bond, spot_curve)

p3 = price_with_credit_spread(bond, risk_free_rate, credit_spread)

p4 = price_with_greenium(bond, risk_free_rate, credit_spread, greenium_bp)

print("EDF Green Bond valuation\n")

print("1️⃣ Price via YTM:", round(p1,2))

print("2️⃣ Price via spot curve:", round(p2,2))

print("3️⃣ Price via credit spread:", round(p3,2))

print("4️⃣ Price with greenium (15bp):", round(p4,2))

EDF Green Bond valuation

1️⃣ Price via YTM: 997.32
2️⃣ Price via spot curve: 1000.94
3️⃣ Price via credit spread: 970.98
4️⃣ Price with greenium (15bp): 978.79
